In [1]:
import os
import random
import pickle
import numpy as np
import pandas as pd

# --- Fix: avoid TF protobuf crash in this environment by preferring standalone keras if present.
# The original code uses only Keras APIs (Model/Layer/Lambda/Adam), so this preserves core logic.
try:
    import tensorflow as tf  # may fail in some Kaggle images due to protobuf mismatch
    from tensorflow import keras as K
    from tensorflow.keras import layers as L
except Exception as e:
    # Fallback to standalone keras; keep tf only for tensor ops if TF import succeeded.
    import keras as K  # type: ignore
    from keras import layers as L  # type: ignore

    tf = None

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error



2026-05-12 07:23:25.396762: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 07:23:25.454533: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778570605.482103      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778570605.491687      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 07:23:25.547867: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Small helper: get a working tf module even if we had to fallback to standalone keras.
# keras 3 uses a backend; but for the few tf ops in losses we need tensorflow.
if tf is None:
    import tensorflow as tf  # if this fails too, the notebook cannot run any TF-based model




In [3]:
def seed_all(seed=20):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


seed_all(20)



In [4]:
# Paths available in this environment (provided by user)
BASE_PATH = "/kaggle/input/osic-pulmonary-fibrosis-progression"
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")
SAMPLE_SUB = os.path.join(BASE_PATH, "sample_submission.csv")

assert os.path.exists(TRAIN_CSV), f"Missing: {TRAIN_CSV}"
assert os.path.exists(TEST_CSV), f"Missing: {TEST_CSV}"
assert os.path.exists(SAMPLE_SUB), f"Missing: {SAMPLE_SUB}"



In [5]:
# --- Fix: correct broken input paths (was referencing non-existent dataset)
train_raw = pd.read_csv(TRAIN_CSV)
raw_test = pd.read_csv(TEST_CSV)
X_prediction = pd.read_csv(SAMPLE_SUB)



In [6]:
ID = "Patient_Week"
PINBALL_QUANTILE = [0.2, 0.50, 0.8]
LAMBDA_LOSS = 0.8
EPOCH = 250
BATCH_SIZE = 128



In [7]:
# Build training features consistent with what prediction-time construction expects:
# ['Weeks','Percent','Age','Sex','Min_week','Base_FVC','Base_week', onehot(smoking)]
# We derive per-patient baseline from the earliest available week in train.
base = (
    train_raw.sort_values(["Patient", "Weeks"])
    .groupby("Patient", as_index=False)
    .first()[["Patient", "Weeks", "FVC", "Percent", "Age", "Sex", "SmokingStatus"]]
    .rename(columns={"Weeks": "Min_week", "FVC": "Base_FVC"})
)

train = train_raw.merge(base, on="Patient", how="left", suffixes=("", "_base"))
train["Base_week"] = train["Weeks"] - train["Min_week"]

# Keep only columns needed downstream; target is FVC from train_raw rows
train = train[
    [
        "Patient",
        "Weeks",
        "FVC",
        "Percent",
        "Age",
        "Sex",
        "SmokingStatus",
        "Min_week",
        "Base_FVC",
        "Base_week",
    ]
].reset_index(drop=True)



In [8]:
C1, C2 = tf.constant(70, dtype="float32"), tf.constant(1000, dtype="float32")


def score(y_true, y_pred):
    # y_pred shape: (n, 3) [q20, q50, q80] after cumulative sum trick
    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]

    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:, 0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt(tf.cast(2.0, dtype=tf.float32))
    metric = (delta / sigma_clip) * sq2 + tf.math.log(sigma_clip * sq2)
    return K.backend.mean(metric)


def qloss(y_true, y_pred):
    qs = PINBALL_QUANTILE
    q = tf.constant(np.array([qs]), dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(q * e, (q - 1) * e)
    return K.backend.mean(v)


def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1 - _lambda) * score(y_true, y_pred)

    return loss




2026-05-12 07:23:34.313519: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [9]:
# Keep eval_score but adapt it to the fitted data_preparation object we will create locally.
def eval_score(y_true, y_pred):
    y_true = (
        tf.cast(y_true, tf.float32) * (data_prep.fvc_max - data_prep.fvc_min)
        + data_prep.fvc_min
    )
    y_pred = (
        tf.cast(y_pred, tf.float32) * (data_prep.fvc_max - data_prep.fvc_min)
        + data_prep.fvc_min
    )

    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]

    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt(tf.cast(2.0, dtype=tf.float32))
    metric = (delta / sigma_clip) * sq2 + tf.math.log(sigma_clip * sq2)
    return -K.backend.mean(metric)




In [10]:
SELECTED_COLUMNS = [
    "Weeks",
    "Percent",
    "Age",
    "Sex",
    "Min_week",
    "Base_FVC",
    "Base_week",
    "_Currently smokes",
    "_Ex-smoker",
    "_Never smoked",
]


def create_model(lambda_loss):
    model_input = K.Input(shape=(len(SELECTED_COLUMNS),))
    x = L.Dense(500, activation="selu", name="dense_to_freeze1")(model_input)
    x = L.Dense(100, activation="selu", name="dense_to_freeze2")(x)
    p1 = L.Dense(3, activation="linear", name="p1")(x)
    p2 = L.Dense(3, activation="selu", name="p2")(x)
    FVC = L.Lambda(lambda z: z[0] + tf.cumsum(z[1], axis=1), name="FVC")([p1, p2])

    model = K.Model(inputs=model_input, outputs=[FVC])

    # --- Fix: Keras API: lr -> learning_rate, and 'decay' argument is no longer supported.
    model.compile(
        optimizer=K.optimizers.Adam(
            learning_rate=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-7, amsgrad=False
        ),
        loss=mloss(lambda_loss),
        metrics=[score],
    )
    return model


model = create_model(LAMBDA_LOSS)
_ = model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_to_freeze1    │ (None, 500)       │      5,500 │ input_layer[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_to_freeze2    │ (None, 100)       │     50,100 │ dense_to_freeze1… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p1 (Dense)          │ (None, 3)         │        303 │ dense_to_freeze2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p2 (Dense)          │ (None, 3)         │        303 │ dense_to_freeze2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ FVC (Lambda)        │ (None, 3)         │          0 │ p1[0][0],         │
│                     │                   │            │ p2[0][0]          │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 56,206 (219.55 KB)

 Trainable params: 56,206 (219.55 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Build prediction frame from sample_submission + test clinicals (baseline row only).
X_prediction["Patient"] = X_prediction["Patient_Week"].str.extract(r"(.*)_.*")
X_prediction["Weeks"] = X_prediction["Patient_Week"].str.extract(r".*_(.*)").astype(int)
X_prediction = X_prediction[["Patient", "Weeks", "Patient_Week"]]

rename_cols = {"Weeks_y": "Min_week", "Weeks_x": "Weeks", "FVC": "Base_FVC"}
X_prediction = (
    X_prediction.merge(raw_test, how="left", on="Patient")
    .rename(columns=rename_cols)[
        [
            "Patient",
            "Weeks",
            "Patient_Week",
            "Min_week",
            "Base_FVC",
            "Percent",
            "Age",
            "Sex",
            "SmokingStatus",
        ]
    ]
    .reset_index(drop=True)
)



In [12]:
X_prediction["Base_week"] = X_prediction["Weeks"] - X_prediction["Min_week"]



In [13]:
# --- Fix: the original notebook expected a pickled data_prep object from another dataset.
# We fit data_preparation here on training data (same semantics) and reuse it for test.
from sklearn.preprocessing import OneHotEncoder as SklearnOneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.exceptions import NotFittedError


class OneHotEncoder(SklearnOneHotEncoder):
    def __init__(self, **kwargs):
        super(OneHotEncoder, self).__init__(**kwargs)
        self.fit_flag = False

    def fit(self, X, **kwargs):
        out = super().fit(X)
        self.fit_flag = True
        return out

    def transform(self, X, categories, index="", name="", **kwargs):
        sparse_matrix = super(OneHotEncoder, self).transform(X)
        new_columns = self.get_new_columns(X=X, name=name, categories=categories)
        d_out = pd.DataFrame(sparse_matrix.toarray(), columns=new_columns, index=index)
        return d_out

    def fit_transform(self, X, categories, index, name, **kwargs):
        self.fit(X)
        return self.transform(X, categories=categories, index=index, name=name)

    def get_new_columns(self, X, name, categories):
        return ["{}_{}".format(name, categories[j]) for j in range(len(categories))]


def standardisation(x, u, s):
    return (x - u) / s


def normalization(x, ma, mi):
    return (x - mi) / (ma - mi)


class data_preparation:
    def __init__(self, bool_normalization=True, bool_standard=False):
        self.enc_sex = LabelEncoder()
        self.enc_smok = LabelEncoder()
        self.onehotenc_smok = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
        self.standardisation = bool_standard
        self.normalization = bool_normalization

    def __call__(self, data_untransformed):
        data = data_untransformed.copy(deep=True)

        try:
            data["Sex"] = self.enc_sex.transform(data["Sex"].values)
            data["SmokingStatus"] = self.enc_smok.transform(
                data["SmokingStatus"].values
            )
            data = pd.concat(
                [
                    data.drop(columns=["SmokingStatus"]),
                    self.onehotenc_smok.transform(
                        data["SmokingStatus"].values.reshape(-1, 1),
                        categories=self.enc_smok.classes_,
                        name="",
                        index=data.index,
                    ).astype(int),
                ],
                axis=1,
            )

            if self.normalization:
                data["Base_week"] = normalization(
                    data["Base_week"], self.base_week_max, self.base_week_min
                )
                data["Base_FVC"] = normalization(
                    data["Base_FVC"], self.fvc_max, self.fvc_min
                )
                data["Percent"] = normalization(
                    data["Percent"], self.base_percent_max, self.base_percent_min
                )
                data["Age"] = normalization(data["Age"], self.age_max, self.age_min)
                data["Weeks"] = normalization(
                    data["Weeks"], self.weeks_max, self.weeks_min
                )
                data["Min_week"] = normalization(
                    data["Min_week"], self.base_week_max, self.base_week_min
                )

        except NotFittedError:
            data["Sex"] = self.enc_sex.fit_transform(data["Sex"].values)
            data["SmokingStatus"] = self.enc_smok.fit_transform(
                data["SmokingStatus"].values
            )
            data = pd.concat(
                [
                    data.drop(columns=["SmokingStatus"]),
                    self.onehotenc_smok.fit_transform(
                        data["SmokingStatus"].values.reshape(-1, 1),
                        categories=self.enc_smok.classes_,
                        name="",
                        index=data.index,
                    ).astype(int),
                ],
                axis=1,
            )

            if self.normalization:
                self.base_week_min = data["Base_week"].min()
                self.base_week_max = data["Base_week"].max()
                data["Base_week"] = normalization(
                    data["Base_week"], self.base_week_max, self.base_week_min
                )

                self.fvc_min = (
                    data["FVC"].min()
                    if "FVC" in data.columns
                    else data["Base_FVC"].min()
                )
                self.fvc_max = (
                    data["FVC"].max()
                    if "FVC" in data.columns
                    else data["Base_FVC"].max()
                )

                self.base_fvc_min = data["Base_FVC"].min()
                self.base_fvc_max = data["Base_FVC"].max()
                data["Base_FVC"] = normalization(
                    data["Base_FVC"], self.fvc_max, self.fvc_min
                )

                self.base_percent_min = data["Percent"].min()
                self.base_percent_max = data["Percent"].max()
                data["Percent"] = normalization(
                    data["Percent"], self.base_percent_max, self.base_percent_min
                )

                self.age_min = data["Age"].min()
                self.age_max = data["Age"].max()
                data["Age"] = normalization(data["Age"], self.age_max, self.age_min)

                self.weeks_min = data["Weeks"].min()
                self.weeks_max = data["Weeks"].max()
                data["Weeks"] = normalization(
                    data["Weeks"], self.weeks_max, self.weeks_min
                )

                # keep same scale for Min_week as Base_week
                data["Min_week"] = normalization(
                    data["Min_week"], self.base_week_max, self.base_week_min
                )

        return data




In [14]:
# Fit on train and transform both train and prediction consistently.
data_prep = data_preparation(bool_normalization=True, bool_standard=False)

train_prep = data_prep(train)
X_pred_prep = data_prep(X_prediction)

# Ensure required one-hot columns exist (handle_unknown may drop unseen categories in one split)
for col in ["_Currently smokes", "_Ex-smoker", "_Never smoked"]:
    if col not in train_prep.columns:
        train_prep[col] = 0
    if col not in X_pred_prep.columns:
        X_pred_prep[col] = 0

# Enforce column order
train = train_prep
X_prediction = X_pred_prep



In [15]:
# Prepare training targets: model expects y_true shaped (n, 1) because score uses y_true[:,0]
y = train_raw.loc[train.index, "FVC"].values.reshape(-1, 1).astype("float32")



In [16]:
NFOLD = 5
kf = KFold(n_splits=NFOLD, shuffle=True, random_state=20)

pe = np.zeros((X_prediction.shape[0], 3), dtype="float32")
pred = np.zeros((train.shape[0], 3), dtype="float32")



In [17]:
cnt = 0
EPOCHS = 200

for tr_idx, val_idx in kf.split(train):
    cnt += 1
    print(f"FOLD {cnt}")

    net = create_model(LAMBDA_LOSS)

    net.fit(
        train.loc[tr_idx, SELECTED_COLUMNS].values,
        y[tr_idx],
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(train.loc[val_idx, SELECTED_COLUMNS].values, y[val_idx]),
        verbose=0,
    )

    print(
        "train",
        net.evaluate(
            train.loc[tr_idx, SELECTED_COLUMNS].values,
            y[tr_idx],
            verbose=0,
            batch_size=BATCH_SIZE,
        ),
    )
    print(
        "val",
        net.evaluate(
            train.loc[val_idx, SELECTED_COLUMNS].values,
            y[val_idx],
            verbose=0,
            batch_size=BATCH_SIZE,
        ),
    )

    pred[val_idx] = net.predict(
        train.loc[val_idx, SELECTED_COLUMNS].values, batch_size=BATCH_SIZE, verbose=0
    )
    pe += (
        net.predict(
            X_prediction[SELECTED_COLUMNS].values, batch_size=BATCH_SIZE, verbose=0
        )
        / NFOLD
    )

    # --- Fix: save the trained fold model (was saving the untrained 'model' variable)
    net.save(f"model_{cnt}.keras")



FOLD 1
train [34.88157653808594, 6.362866401672363]
val [38.50294494628906, 6.499774932861328]
FOLD 2
train [34.55088806152344, 6.492483615875244]
val [34.48831558227539, 6.46766471862793]
FOLD 3
train [35.454978942871094, 6.334602355957031]
val [39.51567840576172, 6.400640964508057]
FOLD 4
train [43.13893127441406, 6.632946491241455]
val [43.3621940612793, 6.629377841949463]
FOLD 5
train [36.52322769165039, 6.406773090362549]
val [39.176700592041016, 6.425817012786865]


In [18]:
sigma_opt = mean_absolute_error(train_raw.loc[train.index, ["FVC"]], pred[:, 1])
unc = pred[:, 2] - pred[:, 0]
sigma_mean = float(np.mean(unc))
print("sigma_opt:", sigma_opt, "sigma_mean:", sigma_mean)



sigma_opt: 115.94722359218078 sigma_mean: 244.94821166992188


In [19]:
X_prediction["FVC1"] = 0.996 * pe[:, 1]
X_prediction["Confidence1"] = pe[:, 2] - pe[:, 0]



In [20]:
subm = X_prediction[["Patient_Week", "FVC1", "Confidence1"]].copy()
subm["FVC"] = 3020.0
subm["Confidence"] = 100.0

subm.loc[~subm.FVC1.isnull(), "FVC"] = subm.loc[~subm.FVC1.isnull(), "FVC1"]

# keep same original logic for confidence selection
if sigma_mean < 70:
    subm["Confidence"] = float(sigma_opt)
else:
    subm.loc[~subm.FVC1.isnull(), "Confidence"] = subm.loc[
        ~subm.FVC1.isnull(), "Confidence1"
    ]



In [21]:
# Override baseline rows in test with known FVC and tiny confidence (as in original code).
otest = pd.read_csv(TEST_CSV)
for i in range(len(otest)):
    key = otest.Patient[i] + "_" + str(otest.Weeks[i])
    subm.loc[subm["Patient_Week"] == key, "FVC"] = float(otest.FVC[i])
    subm.loc[subm["Patient_Week"] == key, "Confidence"] = 0.1



In [22]:
# Final submission
submission = subm[["Patient_Week", "FVC", "Confidence"]].copy()
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", submission.shape)
print(submission.head())

Wrote submission.csv with shape: (1908, 3)
                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2236.817139  164.325928
1  ID00126637202218610655908_-2  2233.906250  167.079834
2  ID00126637202218610655908_-1  2231.028809  170.014893
3   ID00126637202218610655908_0  2228.062500  172.983643
4   ID00126637202218610655908_1  2225.013672  175.979736
